#  News Topic Classifier — BERT Fine-Tuning

---

**What this notebook does:**
1. Installs and imports all required libraries
2. Loads and tokenizes the AG News dataset
3. Fine-tunes `bert-base-uncased` for 4-class news classification
4. Evaluates the model with Accuracy and F1-Score
5. Launches a live Gradio demo — type a headline, get the topic instantly

**Before running:** `Runtime → Change runtime type → T4 GPU`

---


---
## 🔧 Section 1 — Install Libraries

In [ ]:
# Install all required libraries
!pip install transformers datasets torch scikit-learn gradio --quiet


---
##  Section 2 — Import Libraries

All imports in one place so the notebook stays clean and easy to read.


In [ ]:
# Standard libraries
import numpy as np
import torch

# Hugging Face — dataset loading
from datasets import load_dataset

# Hugging Face — model and tokenizer
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
)

# Evaluation metrics
from sklearn.metrics import accuracy_score, f1_score

# Demo interface
import gradio as gr

# ── Confirm GPU ────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅  All libraries imported.")
print(f"    Device : {device.upper()}")


✅  All libraries imported.
    Device : CUDA


---
## Section 3 — Load the AG News Dataset

AG News contains real news headlines across **4 categories**:

| Label | Category |
|---|---|
| 0 | 🌍 World |
| 1 | 🏆 Sports |
| 2 | 💼 Business |
| 3 | 💻 Sci / Tech |

We use a **4 000-sample subset** so training finishes in ~10 minutes on a free Colab GPU.


In [ ]:
# Load full AG News dataset from Hugging Face
dataset = load_dataset("ag_news")

print(f"✅  Dataset loaded.")
print(f"    Full train : {len(dataset['train']):,} samples")
print(f"    Full test  : {len(dataset['test']):,} samples")
print()
print("Sample entry:")
print(dataset["train"][0])


✅  Dataset loaded.
    Full train : 120,000 samples
    Full test  : 7,600 samples

Sample entry:
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


In [ ]:
# Use a small subset so training is fast on a free GPU
# To use the full dataset set TRAIN_SIZE = len(dataset["train"])
TRAIN_SIZE = 4000
TEST_SIZE  = 800

train_data = dataset["train"].shuffle(seed=42).select(range(TRAIN_SIZE))
test_data  = dataset["test"].shuffle(seed=42).select(range(TEST_SIZE))

print(f"✅  Subset created.")
print(f"    Train : {len(train_data)} samples")
print(f"    Test  : {len(test_data)} samples")


✅  Subset created.
    Train : 4000 samples
    Test  : 800 samples


---
##  Section 4 — Tokenize the Dataset

The tokenizer converts raw text into numbers that BERT can read.

- **`truncation=True`** — cuts text that is too long
- **`padding=True`** — pads short text so all sequences are the same length
- **`max_length=128`** — 128 tokens is enough for a news headline


In [ ]:
# Load the tokenizer for bert-base-uncased
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation = True,
        padding    = True,
        max_length = 128,
    )

# Apply tokenizer to both splits
train_data = train_data.map(tokenize, batched=True)
test_data  = test_data.map(tokenize, batched=True)

print("✅  Tokenization complete.")


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

✅  Tokenization complete.


In [ ]:
# Rename "label" → "labels"  (Hugging Face Trainer expects this name)
train_data = train_data.rename_column("label", "labels")
test_data  = test_data.rename_column("label", "labels")

# Tell Hugging Face to return PyTorch tensors
train_data.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_data.set_format("torch",  columns=["input_ids", "attention_mask", "labels"])

print("✅  Dataset formatted for PyTorch training.")


✅  Dataset formatted for PyTorch training.


---
##  Section 5 — Load the BERT Model

We load `bert-base-uncased` with a classification head on top.
Hugging Face automatically adds a linear layer that maps BERT's output → 4 class scores.


In [ ]:
# Load bert-base-uncased with a 4-class classification head
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels = 4,
)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅  Model loaded : bert-base-uncased")
print(f"    Parameters  : {total_params:.1f}M")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅  Model loaded : bert-base-uncased
    Parameters  : 109.5M


---
##  Section 6 — Fine-Tune the Model

We define simple training settings and let the Hugging Face `Trainer` handle
the full training loop — no manual forward/backward passes needed.

| Setting | Value |
|---|---|
| Epochs | 2 |
| Batch size | 16 |
| Learning rate | 2e-5 |
| Evaluation | After every epoch |


In [ ]:
# Compute accuracy and F1 after each epoch
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy" : round(accuracy_score(labels, preds), 4),
        "f1"       : round(f1_score(labels, preds, average="weighted"), 4),
    }
print("✅  Metrics ready.{accuracy, f1}")

✅  Metrics ready.{accuracy, f1}


In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Training configuration
training_args = TrainingArguments(
    output_dir                  = "./saved_model",
    num_train_epochs            = 10,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    do_eval                     = True, # Enable evaluation
    eval_steps                  = 250,  # Evaluate every 250 steps (~1 epoch)
    save_steps                  = 250,  # Save every 250 steps (~1 epoch)
    # Removed evaluation_strategy and save_strategy as they cause TypeError
    # Removed load_best_model_at_end as it conflicts with implicit strategies
    logging_steps               = 50,
    report_to                   = "none",
)

# Create the Trainer
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_data,
    eval_dataset    = test_data,
    compute_metrics = compute_metrics,
)

print("✅  Trainer ready.")

# Evaluate before training
print("🚀  Evaluating model before training...")
initial_results = trainer.evaluate()
print(f"   Initial Accuracy : {initial_results['eval_accuracy'] * 100:.2f}%")
print(f"   Initial F1 Score : {initial_results['eval_f1'] * 100:.2f}%")
print(f"   Initial Eval Loss: {initial_results['eval_loss']:.4f}")

print("🚀  Starting training — ~10 min on T4 GPU ...")
print()

trainer.train()

print()
print("✅  Training complete!")

✅  Trainer ready.
🚀  Evaluating model before training...


   Initial Accuracy : 25.50%
   Initial F1 Score : 10.66%
   Initial Eval Loss: 1.4560
🚀  Starting training — ~10 min on T4 GPU ...



Step,Training Loss
50,1.115110
100,0.471096
150,0.409939
200,0.315285
250,0.335283
300,0.209967
350,0.271431
400,0.223397
450,0.247109
500,0.218864


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅  Training complete!


---
##  Section 7 — Evaluate the Model

We run a final evaluation on the held-out test set and display the results.


In [ ]:
# Evaluate on the test set
results = trainer.evaluate()

print()
print("=" * 40)
print("   EVALUATION RESULTS")
print("=" * 40)
print(f"   Accuracy  : {results['eval_accuracy'] * 100:.2f}%")
print(f"   F1 Score  : {results['eval_f1'] * 100:.2f}%")
print(f"   Eval Loss : {results['eval_loss']:.4f}")
print("=" * 40)



   EVALUATION RESULTS
   Accuracy  : 91.13%
   F1 Score  : 91.16%
   Eval Loss : 0.5484


---
##  Section 8 — Save the Model

We save the fine-tuned model and tokenizer so the Gradio demo can load them.


In [ ]:
# Save model and tokenizer
model.save_pretrained("./saved_model")
tokenizer.save_pretrained("./saved_model")

print("✅  Model saved to ./saved_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅  Model saved to ./saved_model


---
##  Section 9 — Live Demo with Gradio

Type any news headline and the model will predict its topic with confidence scores.
The public link is valid for **72 hours**.


In [ ]:
# Label mapping
LABELS = {0: "🌍 World", 1: "🏆 Sports", 2: "💼 Business", 3: "💻 Sci/Tech"}

# Load saved model for inference
model.eval()

def predict(headline: str) -> dict:
    if not headline.strip():
        return {label: 0.0 for label in LABELS.values()}

    inputs = tokenizer(
        headline,
        return_tensors = "pt",
        truncation     = True,
        padding        = True,
        max_length     = 128,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1).squeeze().tolist()
    return {LABELS[i]: round(probs[i], 4) for i in range(4)}


# Example headlines
examples = [
    ["NASA discovers water ice on the Moon's surface"],
    ["Apple reports record $120 billion quarterly revenue"],
    ["England beats Australia in dramatic World Cup final"],
    ["UN calls for ceasefire as conflict enters third week"],
    ["Google releases new AI model that codes better than humans"],
    ["Oil prices surge after OPEC cuts production"],
]

# Build the Gradio interface
demo = gr.Interface(
    fn          = predict,
    inputs      = gr.Textbox(
                    lines       = 2,
                    placeholder = "Enter a news headline...",
                    label       = "News Headline",
                  ),
    outputs     = gr.Label(num_top_classes=4, label="Predicted Topic"),
    title       = "🗞️ News Topic Classifier — BERT",
    description = "Fine-tuned BERT model that classifies news headlines into World · Sports · Business · Sci/Tech",
    examples    = examples,
    theme       = gr.themes.Soft(),
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://274dd0769f8ac93082.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
